# DefectosCafeVerde — DVF1 pair-transition review

Validation-only review of the completed seed-42 fuser. It reports rescue/regression transitions, exact McNemar, and 10,000 class-stratified physical-pair bootstrap iterations. No model is trained and test is not accessed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import importlib, json, os, shutil, subprocess, sys
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'; BRANCH='codex/defectoscafeverde-dual-view-fusion'
relative=Path('experiments/defectoscafeverde-dvf1-v1/DVF1_seed42_result.json')
roots=[p for p in [Path('/content/drive/MyDrive'),*Path('/content/drive/.shortcut-targets-by-id').glob('*')] if p.exists()]
matches=sorted({path for root in roots for path in root.rglob(relative.name) if path.as_posix().endswith(relative.as_posix())})
if not matches: raise FileNotFoundError(f'DVF1 result tidak ditemukan di shared Drive: {relative}')
RESULT=matches[0]; PROJECT=RESULT.parents[2]; OUT=RESULT.parent
CHECKPOINT=OUT/'DVF1_last.pt'; CACHE=OUT/'val_pair_cache.pt'
for required in (RESULT,CHECKPOINT,CACHE):
    if not required.is_file(): raise FileNotFoundError(required)
print('PROJECT:',PROJECT); print('RESULT:',RESULT); print('CHECKPOINT:',CHECKPOINT); print('CACHE:',CACHE)


In [ ]:
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True,cwd=WORK)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
check=subprocess.run([sys.executable,'-m','pytest','-q','tests/test_defectoscafeverde_dvf1_review.py'],cwd=REPO,text=True,capture_output=True)
print(check.stdout)
if check.returncode: print(check.stderr); raise RuntimeError('DVF1 review tests gagal')


In [ ]:
SUMMARY=OUT/'DVF1_seed42_review.json'; RECORDS=OUT/'DVF1_seed42_pair_transitions.json'; LOG=OUT/'DVF1_seed42_review.log'
command=[sys.executable,'-u','-m','coffee_detector.analysis.defectoscafeverde_dvf1_review','--result',str(RESULT),'--checkpoint',str(CHECKPOINT),'--validation-cache',str(CACHE),'--output',str(SUMMARY),'--records-output',str(RECORDS)]
print('MENJALANKAN VALIDATION-ONLY DVF1 REVIEW',flush=True)
with LOG.open('w',encoding='utf-8') as stream:
    process=subprocess.run(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
print(LOG.read_text(errors='replace'))
if process.returncode: raise RuntimeError(f'Review gagal: {process.returncode}; log={LOG}')
if not SUMMARY.is_file(): raise FileNotFoundError(SUMMARY)


In [ ]:
from IPython.display import display
import pandas as pd
result=json.loads(SUMMARY.read_text())
transition_rows=[{'transition':key,'pairs':value} for key,value in result['transitions'].items()]
display(pd.DataFrame(transition_rows))
class_rows=[{'class_name':name,**values} for name,values in result['classwise'].items()]
display(pd.DataFrame(class_rows).sort_values('net_correct_pairs',ascending=False))
bootstrap_rows=[{'metric':name,**values} for name,values in result['bootstrap']['metrics'].items()]
display(pd.DataFrame(bootstrap_rows).style.format({'point_delta':'{:+.2%}','ci95_low':'{:+.2%}','ci95_high':'{:+.2%}','probability_positive':'{:.2%}','probability_nonnegative':'{:.2%}'}))
print('NET CORRECT:',result['net_correct_pairs'],'| MCNEMAR P:',result['mcnemar_exact_two_sided_p'])
print('CALIBRATION:',result['endpoint_calibration'])
print('CRITERIA:',result['criteria'])
print('DECISION:',result['decision'],'| NEXT:',result['next'])
print('TRAINING: False | TEST:',result['test_images_accessed'])
print('SUMMARY:',SUMMARY,'| RECORDS:',RECORDS)
print('Kirim tiga tabel, NET CORRECT, MCNEMAR P, CALIBRATION, CRITERIA, DECISION, dan NEXT. Jangan training atau membuka test.')
